# 12 - Train a Retrace(λ) DQN Model Offline

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`, but trains with `RetraceObjective` ([Munos et al., 2016](https://arxiv.org/abs/1606.02647)) instead of one-step `DqnObjective`:

1. Load previously collected `Datastore` streams from the Hub.
2. Build a `DataLoader` that samples fixed-length sequences from those streams.
3. Assemble a `Model` from an embedder, a backbone, an action-value head **and a behavior head**.
4. Train with `RetraceObjective` and save with `push_model_to_hub`.

Retrace(λ) is off-policy, return-based Q-learning. The TD target of each transition is the delayed one-step expected backup plus a trace of later TD errors, each scaled by the product of **truncated importance ratios** `c_s = λ · min(1, π(a_s|s_s) / μ(a_s|s_s))`. `π` is the target policy — `softmax(Q / temperature)` over the delayed Q (Q as logits), the same convention as `model.get_action(temperature=)` — and `μ` is the behavior policy that produced the data. Near-on-policy transitions keep the full λ-return; strongly off-policy actions cut the trace; the clip at `1` keeps the variance bounded, so `λ = 1` is safe (the paper's Atari setting).

The dataset stores no behavior probabilities. `μ` is **learned**: a second `ClassificationHead` under the `behavior` key outputs logits whose `log_softmax` is `log μ(·|s)`; it is fit by negative log-likelihood of the actions in the data (inside the same objective call), and the same distribution at each step is the `μ(·|s)` the trace uses. Because it sees the same in-context history as the Q head, it can track a behavior policy that changes along a task — such as the per-episode oracle ramp in `01_collect_dataset.ipynb`.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `15_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    to_device,
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import RetraceObjective, boundary_discount
from mouse_core.models import Model, Polyak, push_model_to_hub
from mouse_core.models.backbone import TransformerBackbone
from mouse_core.models.heads import ClassificationHead, RegressionHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-retrace-offline"          # Hugging Face model repo for push_model_to_hub
TOKENIZER_ID = "mouse-example-tokenizer-retrace-offline"  # Hugging Face tokenizer repo (separate from MODEL_ID)
PRETRAINED = "Qwen/Qwen3-0.6B"                  # HF checkpoint for Tokenizer and backbone
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
TD_LAMBDA = 1.0                               # Retrace λ (1.0: traces are cut only by min(1, π/μ))
TARGET_TEMPERATURE = 0.1                      # softmax temperature of the target policy π = softmax(Q / T) on the delayed Q (0 = greedy)
BEHAVIOR_WEIGHT = 1.0                         # weight of the behavior head's NLL loss that learns μ
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0001                     # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_BACKBONE = 0.01                    # delayed backbone interpolation


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → backbone.embed`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` shares draws within one sampled sequence). Each window gets its own `reseed` generation, so the same index on two rollouts draws two seeds. Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(stages=(augmenter, tokenizer))`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `15_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "text",
            "input_field": "action",
            "format": "{field}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "format": ",{field}",
        },
        {
            "type": "text",
            "input_field": "reward",
            "format": ",r={field:g}",
            "skip": 0.0,
            "format_skipped": "",
        },
        {
            "type": "text",
            "input_field": "episode_done",
            "format": ",d={field}",
            "skip": 0,
            "format_skipped": "",
        },
        {
            "type": "text",
            "output_field": "value",
            "format": "\n",
            "max_tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
    group_prefix="action,observation,r=reward,d=done\n",
    pretrained=PRETRAINED,
)

train_transform = compose(stages=(augmenter, tokenizer))

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)



## Build The Model

A Mouse Core `Model` has a backbone and heads:

- `TransformerBackbone(pretrained=...)` loads the checkpoint including `embed_tokens`, looks up the packed `__text__` ids, and runs the decoder. Step templates and field packing live on `Tokenizer` only. Three arguments are required and describe how it runs on this machine rather than what it is, so they are not saved with the model and `load_model` asks for them again: `train_kernel` for the uncached forward (`"flex"`, block-sparse FlexAttention), `decode_kernel` for cached decode (`"flex"`, paged FlexAttention) and `dtype` for the base weights (`torch.float32` so the whole backbone trains in fp32). `model.to(device)` only moves. `use_norm` (required, saved with the model) keeps (`True`) or drops (`False`) the final RMSNorm.
- `RegressionHead` predicts one value per discrete action. `use_norm` (required) keeps (`True`) or drops (`False`) the head's input RMSNorm.
- `ClassificationHead` predicts one logit per discrete action. Here it is the **behavior head**: its softmax is the learned `μ(·|s)`. It reads the same pooled features as the Q head.

The backbone exposes `hidden_dim`, and the head uses that same value so the pieces connect cleanly.

`Tokenizer` text fields match `02`: comma-separated action / observation, `r=` / `d=` when nonzero, and a const newline readout flagged `head_output: True`.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache. Heads are passed as a dict with caller-chosen keys — `action_value` for Q and `behavior` for μ — and `action_source="action_value"` tells `get_action` to read the Q head, so the behavior head never picks actions at inference.


In [ ]:
backbone = TransformerBackbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    use_norm=True,
    pretrained=PRETRAINED,
)


q_head = RegressionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
    use_norm=True,
)

behavior_head = ClassificationHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=1.0,
    use_norm=True,
)

model = Model(
    backbone=backbone,
    heads={"action_value": q_head, "behavior": behavior_head},
    action_source="action_value",
    reasoner=None,
).train().to(device)
print(model)



## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step head predictions.
3. `objective(objective_data=objective_data, predictions=q, delayed_predictions=q_target, behavior_predictions=mu)` computes the Retrace loss and metrics. Pass the tensors named in `heads=` (`"action_value"` and `"behavior"`).
4. `AdamW` updates weights. The backbone and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights.
5. Delayed Q comes from the delayed model: `delayed_model = model.copy(heads=(q_head,), backbone=True, reasoner=False)` copies the heads the target reads and the fp32 backbone (including token embeddings). The `behavior` head is left out: Retrace reads `μ` from the online head only, so the delayed model neither runs it nor Polyak-interpolates it. After the online forward, `delayed_model(inputs)` runs the same `TokenBatch` through the delayed model under `torch.no_grad()`. `polyak.update(tau_heads=POLYAK_TAU_HEADS, tau_backbone=POLYAK_TAU_BACKBONE)` interpolates each copied section toward the online model after the optimizer step: `0` keeps it frozen, `1` copies the online weights (no delay). Every interpolated parameter is fp32, so a small `tau` is never rounded away.

`RetraceObjective` takes `td_lambda` (`TD_LAMBDA`), `temperature` (`TARGET_TEMPERATURE`), and `behavior_weight` (`BEHAVIOR_WEIGHT`), all required. Along a run the target is

`G_i = r_i + γ_i · ( V_π(s_{i+1}) + c_{i+1} · (G_{i+1} − Q(s_{i+1}, a_{i+1})) )`, with `V_π = E_π Q + temperature · H[π]` and `c = λ · min(1, π/μ)`,

where every target quantity — `V_π = E_π Q + temperature · H[π]`, `Q(s', a')`, and `π = softmax(Q / temperature)` itself — comes from the delayed model (`π` over the head's raw Q treated as logits, so `temperature` is the SAC `α` and means the same as in `get_action`), and `μ(·|s)` is the online behavior head's softmax, detached. `temperature=0` is the greedy target policy, i.e. Watkins's Q(λ), with `V_π = E_π Q`; a higher temperature flattens `π`, cuts fewer traces, and raises the soft value. `td_lambda=0` is the expected one-step target. The returned loss is `td_loss + behavior_weight · behavior_loss`, where `behavior_loss` is the behavior head's negative log-likelihood of the action taken from each step, `-log μ(a|s)`. `behavior_weight=0` drops the NLL from the loss; the head is still required and its softmax is still `μ`. Metrics: `td_loss`, `behavior_loss`, `behavior_prob_mean` (μ of the taken action — rises as the behavior head fits the data), `retrace_ratio_mean` (mean `min(1, π/μ)`; `1` is on-policy, near `0` means the traces are cut everywhere and the target is one-step), and `entropy` when `temperature > 0`.

`reward` is called with the unpacked `objective_data` columns. `boundary_reward` is the standard lookup: `(scale × episode scale × task scale) * reward + shift + episode shift + task shift`. Scale extras are `1.0` and shift extras are `0.0` when the matching code is `0`. `reward=None` / `value=None` / `discount=None` skip that callable. When a task ends both extras fire. `value` is called as `value(value=..., **objective_data)`. `boundary_value` is the same lookup on online and delayed Q. `discount` is called with the unpacked `objective_data` columns exactly as in `DqnObjective`. The gamma at `i+1` multiplies both the bootstrap and the continued trace, so a `0` gamma ends the trace and a non-zero truncation gamma carries it, discounted. The trace never crosses a run break (`sequence_id` / `grouping_field`). Required `bootstrap_cutoff`: `True` adds `V` where the continuation leaves the sampled run (end of the batch, or a `sequence_id` / `grouping_field` break — a chunk boundary, time limit, or truncation whose rest was not sampled). `False` omits that value. A bootstrap at a state that still has later in-run steps is unchanged, and a true terminal is unchanged either way because its γ already multiplies the value.


In [ ]:
optimizer = AdamW(
    params=model.parameters(),
    lr=1e-05,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
delayed_model = model.copy(heads=(q_head,), backbone=True, reasoner=False)
polyak = Polyak(online=model, delayed=delayed_model)
objective = RetraceObjective(
    td_lambda=TD_LAMBDA,
    temperature=TARGET_TEMPERATURE,
    behavior_weight=BEHAVIOR_WEIGHT,
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_episode_terminal=1.0,
        gamma_episode_truncated=1.0,
        gamma_task_terminal=0.0,
        gamma_task_truncated=0.0,
    ),
    grouping_field="task_index",
    bootstrap_cutoff=True,
)

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, optimizer: AdamW, objective: RetraceObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        out = model(inputs)
        with torch.no_grad():
            delayed_out = delayed_model(inputs)
        loss, metrics = objective(
            objective_data=to_device(data=objective_data, device=device), predictions=out.predictions["action_value"], delayed_predictions=delayed_out.predictions["action_value"], behavior_predictions=out.predictions["behavior"],
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak.update(
            tau_heads=POLYAK_TAU_HEADS,
            tau_backbone=POLYAK_TAU_BACKBONE,
        )
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `15_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        delayed_model=delayed_model,
        polyak=polyak,
        optimizer=optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    print(
        f"cycle={cycle} train  loss={loss.item():.4f}  td={metrics['td_loss']:.4f}  "
        f"bc={metrics['behavior_loss']:.4f}  mu={metrics['behavior_prob_mean']:.3f}  "
        f"ratio={metrics['retrace_ratio_mean']:.3f}  q={metrics['q_values_mean']:.3f}"
    )
loader.close()


## Push To The Hub

`push_model_to_hub` uploads the model to `MODEL_ID` and the tokenizer packing spec to a different repo (`TOKENIZER_ID`). Later, `load_model` reconstructs the `Model` and `load_tokenizer` on the tokenizer repo reloads the packing spec — formats, skips, and `head_output` cannot be recovered from the embedder alone.


In [ ]:
model.eval().to("cpu")
model_url, tokenizer_url = push_model_to_hub(model=model, tokenizer=tokenizer, repo_id=MODEL_ID, tokenizer_repo_id=TOKENIZER_ID, private=False, clear=True)
print(f"Pushed model to {model_url}\nPushed tokenizer to {tokenizer_url}")